# Step 4 — Text Assembly

Pulls all downloaded text passages per person and assembles them into a structured JSON document with source attribution preserved. Each assembled document contains:

- **Person metadata** — canonical name, known states, active years, name variants
- **Source-tagged text blocks** — each passage labeled with its origin (source type, title, URL, item ID, page)
- **Keyword-filtered flag** — distinguishes passages that matched railroad/financial keywords from general mentions
- **Enrichment data** — birth/death years, Wikidata QID if available

Output is stored in `assembled/` — one JSON file per person. This directory is the sole input for Step 5 extraction.

Preserving source attribution is essential: Step 5 quotes source passages verbatim, and verifying extracted claims requires tracing them back to the original text.

In [6]:
import sys, json
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('.')))
from db import get_connection, set_progress, pending_persons, get_all_persons, name_variants_for

conn = get_connection()
STEP = '4_text_assembly'
ASSEMBLED_DIR = Path('assembled')
ASSEMBLED_DIR.mkdir(exist_ok=True)

## Assembly function

In [7]:
import re

FINANCIAL_KEYWORDS = re.compile(
    r'railroad|railway|rail road|r\.\s?r\.|'
    r'stock(?:holder)?|director|president of|officer|treasurer|secretary of|'
    r'bank|trust(?:ee)?|bond|mortgage|'
    r'investor|creditor|debtor|land grant',
    re.IGNORECASE,
)

OBITUARY_KEYWORDS = re.compile(
    r'died|death|deceased|funeral|obituary|demise|buried|surviving',
    re.IGNORECASE,
)

WINDOW = 500          # chars on each side of the best anchor
MAX_PASSAGE = 3000    # hard cap after trimming


def _find_name_positions(text_lower, canonical_name):
    """Return list of (start, end) for every occurrence of the person's last name."""
    parts = canonical_name.lower().split()
    last = parts[-1] if parts else ""
    if len(last) < 3:
        return []
    positions = []
    start = 0
    while True:
        idx = text_lower.find(last, start)
        if idx == -1:
            break
        positions.append((idx, idx + len(last)))
        start = idx + 1
    return positions


def _keyword_positions(text, pattern):
    """Return list of match start positions."""
    return [m.start() for m in pattern.finditer(text)]


def trim_passage(text, canonical_name, window=WINDOW, max_len=MAX_PASSAGE):
    """
    Trim a long passage to a window around the best name+keyword co-occurrence.
    Returns (trimmed_text, was_trimmed).
    """
    if len(text) <= max_len:
        return text, False

    text_lower = text.lower()
    name_pos = _find_name_positions(text_lower, canonical_name)
    kw_pos = _keyword_positions(text, FINANCIAL_KEYWORDS)
    obit_pos = _keyword_positions(text, OBITUARY_KEYWORDS)
    all_kw = kw_pos + obit_pos

    if not name_pos:
        # Name not found — take the first max_len chars
        return text[:max_len], True

    # Find the name occurrence closest to any keyword
    best_anchor = name_pos[0][0]  # fallback: first name occurrence
    best_dist = float('inf')
    for ns, ne in name_pos:
        for kp in all_kw:
            d = abs(ns - kp)
            if d < best_dist:
                best_dist = d
                best_anchor = ns

    # Build window around best anchor
    start = max(0, best_anchor - window)
    end = min(len(text), best_anchor + window)

    # Expand to max_len if room allows
    remaining = max_len - (end - start)
    if remaining > 0:
        expand_before = min(start, remaining // 2)
        start -= expand_before
        remaining -= expand_before
        end = min(len(text), end + remaining)

    trimmed = text[start:end]
    return trimmed, True


def score_passage(block, canonical_name):
    """
    Score a text block for relevance. Higher = more relevant to extraction.
    Returns float score.
    """
    text = (block.get("passage") or block.get("context") or "").lower()
    if not text:
        return -1.0

    name_pos = _find_name_positions(text, canonical_name)
    fin_kw = _keyword_positions(text, FINANCIAL_KEYWORDS)
    obit_kw = _keyword_positions(text, OBITUARY_KEYWORDS)

    score = 0.0

    # Name presence (baseline requirement)
    if not name_pos:
        return -1.0
    score += min(len(name_pos), 5) * 1.0  # up to 5 pts for name frequency

    # Financial keyword count (most valuable)
    score += min(len(fin_kw), 10) * 3.0

    # Obituary keyword count (useful but less targeted)
    score += min(len(obit_kw), 5) * 1.5

    # Proximity bonus: closest name-to-financial-keyword distance
    if name_pos and fin_kw:
        min_dist = min(abs(ns - kp) for ns, _ in name_pos for kp in fin_kw)
        if min_dist < 100:
            score += 15.0
        elif min_dist < 300:
            score += 10.0
        elif min_dist < 600:
            score += 5.0

    # Penalise very long passages (likely full-page OCR dumps)
    if len(text) > 8000:
        score -= 3.0

    # Bonus for keyword-filtered flag from scraping step
    if block.get("is_keyword_filtered"):
        score += 2.0

    return score


print("Trimming and scoring functions defined.")

Trimming and scoring functions defined.


In [8]:
def assemble_person_document(conn, research_id):
    """
    Assemble a structured document for one person from all their downloaded text passages.
    Trims long passages and scores each block for relevance.
    Returns a dict ready for JSON serialization.
    """
    # Person metadata
    person = conn.execute(
        "SELECT * FROM persons WHERE research_id=?", (research_id,)
    ).fetchone()

    canonical_name = person["canonical_name"]

    enrichment = conn.execute(
        """SELECT birth_year, death_year, birth_place, death_place,
                  wikidata_qid, external_ids, occupations
           FROM enrichment WHERE research_id=? ORDER BY enriched_at LIMIT 1""",
        (research_id,)
    ).fetchone()

    # All text passages, with source info
    passages = conn.execute(
        """SELECT t.id as text_id, t.source_type, t.source_url, t.source_title,
                  t.passage_text, t.context_text, t.is_keyword_filtered,
                  t.keyword_match, t.page_num,
                  sd.title as discovered_title, sd.item_id, sd.relevance_note
           FROM texts_downloaded t
           LEFT JOIN sources_discovered sd ON t.source_id = sd.id
           WHERE t.research_id=?
           ORDER BY t.is_keyword_filtered DESC, t.source_type, t.id""",
        (research_id,)
    ).fetchall()

    # Build and filter text blocks
    text_blocks = []
    trimmed_count = 0
    for p in passages:
        raw_passage = p["passage_text"] or ""
        raw_context = p["context_text"] or ""

        # Trim long passages to a window around name+keyword co-occurrence
        passage, was_trimmed_p = trim_passage(raw_passage, canonical_name)
        context, was_trimmed_c = trim_passage(raw_context, canonical_name) if raw_context else ("", False)
        if was_trimmed_p or was_trimmed_c:
            trimmed_count += 1

        block = {
            "text_id": p["text_id"],
            "source_type": p["source_type"],
            "source_title": p["source_title"] or p["discovered_title"] or "",
            "source_url": p["source_url"] or "",
            "item_id": p["item_id"] or "",
            "page": p["page_num"] or "",
            "is_keyword_filtered": bool(p["is_keyword_filtered"]),
            "keywords_matched": p["keyword_match"] or "",
            "passage": passage,
            "context": context,
        }

        # Score for relevance (used by step 5 to prioritise blocks)
        block["relevance_score"] = score_passage(block, canonical_name)
        text_blocks.append(block)

    # Sort by relevance score descending so step 5 picks the best ones first
    text_blocks.sort(key=lambda b: -b["relevance_score"])

    doc = {
        "research_id": research_id,
        "person_id": person["person_id"],
        "canonical_name": canonical_name,
        "name_variants": json.loads(person["name_variants"] or "[]"),
        "known_states": json.loads(person["known_states"] or "[]"),
        "known_newspapers": json.loads(person["known_newspapers"] or "[]"),
        "first_active_year": person["first_active_year"],
        "last_active_year": person["last_active_year"],
        "is_ambiguous": bool(person["is_ambiguous"]),
        "enrichment": {
            "birth_year": enrichment["birth_year"] if enrichment else None,
            "death_year": enrichment["death_year"] if enrichment else None,
            "birth_place": enrichment["birth_place"] if enrichment else None,
            "death_place": enrichment["death_place"] if enrichment else None,
            "wikidata_qid": enrichment["wikidata_qid"] if enrichment else None,
            "external_ids": json.loads(enrichment["external_ids"] or "{}") if enrichment else {},
        } if enrichment else {},
        "source_summary": {
            "total_passages": len(text_blocks),
            "keyword_filtered": sum(1 for b in text_blocks if b["is_keyword_filtered"]),
            "general_mentions": sum(1 for b in text_blocks if not b["is_keyword_filtered"]),
            "trimmed_passages": trimmed_count,
            "sources_by_type": {},
        },
        "text_blocks": text_blocks,
    }

    # Count by source type
    for block in text_blocks:
        st = block["source_type"] or "unknown"
        doc["source_summary"]["sources_by_type"][st] = \
            doc["source_summary"]["sources_by_type"].get(st, 0) + 1

    return doc


def assemble_and_save(conn, research_id):
    """Assemble and save to assembled/{research_id}.json. Returns passage count."""
    doc = assemble_person_document(conn, research_id)
    out_path = ASSEMBLED_DIR / f"{research_id}.json"
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)
    return doc["source_summary"]["total_passages"]

## Run assembly

In [9]:
todo = pending_persons(conn, STEP)
print(f"{len(todo)} persons pending for step '{STEP}'")

persons_with_text = 0
persons_without_text = 0
total_passages_assembled = 0

from tqdm import tqdm
for research_id in tqdm(todo):
    try:
        count = assemble_and_save(conn, research_id)
        total_passages_assembled += count
        if count > 0:
            persons_with_text += 1
        else:
            persons_without_text += 1
        set_progress(conn, research_id, STEP, 'done', result_count=count)
    except Exception as e:
        print(f"\n  Error for research_id={research_id}: {e}")
        set_progress(conn, research_id, STEP, 'failed', error_msg=str(e))

print(f"\nAssembly complete:")
print(f"  Persons with text: {persons_with_text}")
print(f"  Persons with no text: {persons_without_text}")
print(f"  Total passages assembled: {total_passages_assembled}")
print(f"  Files written to: {ASSEMBLED_DIR.resolve()}")

0 persons pending for step '4_text_assembly'


0it [00:00, ?it/s]


Assembly complete:
  Persons with text: 0
  Persons with no text: 0
  Total passages assembled: 0
  Files written to: C:\Users\samwt\Documents\ExtendedEssay\owner_bio_scraping\assembled


## Review assembled documents

In [10]:
import os

files = sorted(ASSEMBLED_DIR.glob("*.json"))
print(f"Assembled documents: {len(files)}\n")

# Load and summarize
summaries = []
for f in files:
    with open(f, encoding='utf-8') as fp:
        doc = json.load(fp)
    summaries.append({
        "research_id": doc["research_id"],
        "name": doc["canonical_name"],
        "total": doc["source_summary"]["total_passages"],
        "keyword": doc["source_summary"]["keyword_filtered"],
        "states": ", ".join(doc["known_states"][:2]),
        "years": f"{doc['first_active_year'] or '?'}\u2013{doc['last_active_year'] or '?'}",
    })

summaries.sort(key=lambda x: -x["total"])

print(f"{'Name':<35} {'Total':>6} {'Kw-filt':>8} {'States':<20} {'Years'}")
print("-" * 80)
for s in summaries[:30]:
    if s["total"] > 0:
        print(f"{s['name']:<35} {s['total']:>6} {s['keyword']:>8} {s['states']:<20} {s['years']}")

persons_with_text = sum(1 for s in summaries if s["total"] > 0)
print(f"\n{persons_with_text}/{len(summaries)} persons have at least one text passage")
print(f"Average passages (non-zero): {sum(s['total'] for s in summaries if s['total']>0)/max(1,persons_with_text):.1f}")

Assembled documents: 570

Name                                 Total  Kw-filt States               Years
--------------------------------------------------------------------------------
H. Smith                              4692      142 PENNSYLVANIA         1869–1885
W. Thompson                           2104      244 GEORGIA              1869–1882
W. Scott                              1670        0 SOUTH CAROLINA       1879–1879
O. Palmer                             1066        0 MICHIGAN             1890–1890
E. Wells                              1010        0 MARYLAND             1876–1879
Horace Greeley                         604      177 NEW YORK             1869–1871
W. T. Thompson                         555      237 GEORGIA              1879–1880
John T. Smith                          489        0                      1871–1872
L. Knapp                               440        0                      1869–1877
James Gordon Bennett                   400      220 NEW YORK       

## Sample document

Preview one assembled document to verify structure before running Step 5.

In [11]:
# Preview the person with the most text
if summaries and summaries[0]["total"] > 0:
    sample_id = summaries[0]["research_id"]
    with open(ASSEMBLED_DIR / f"{sample_id}.json", encoding='utf-8') as f:
        sample = json.load(f)

    print(f"Sample: {sample['canonical_name']} (research_id={sample_id})")
    print(f"  States: {', '.join(sample['known_states'])}")
    print(f"  Newspapers: {', '.join(sample['known_newspapers'][:3])}")
    print(f"  Total passages: {sample['source_summary']['total_passages']}")
    print(f"  By type: {sample['source_summary']['sources_by_type']}")
    print(f"\nFirst keyword-filtered passage:")
    for block in sample["text_blocks"]:
        if block["is_keyword_filtered"]:
            print(f"  Source: {block['source_title'] or block['source_url']}")
            print(f"  Keywords: {block['keywords_matched']}")
            print(f"  Context (first 500 chars):\n    {block['context'][:500]}")
            break
else:
    print("No assembled documents with text yet. Run Steps 3a/3b first.")

Sample: H. Smith (research_id=65)
  States: PENNSYLVANIA
  Newspapers: Intelligencer
  Total passages: 4692
  By type: {'chronicling_america': 142, 'internet_archive': 4550}

First keyword-filtered passage:
  Source: Image 11 of New-York tribune (New York [N.Y.]), February 23, 1921
  Keywords: bank, death, died, estate, funeral, in memoriam, president of, railroad, railway, secretary of
  Context (first 500 chars):
    
